In [1]:
import os
os.environ['KERAS_BACKEND'] = 'torch'

In [2]:
import keras
import torch
import tables
import datasets
import numpy as np
import transformers
import tqdm.notebook as tqdm
import sklearn.model_selection
import matplotlib.pyplot as plt

In [3]:
train = datasets.load_dataset('wangrongsheng/ag_news', split='train')
test  = datasets.load_dataset('wangrongsheng/ag_news', split='test')

# jinaai

In [ ]:
jinaai_model = transformers.AutoModel.from_pretrained('jinaai/jina-embeddings-v3', trust_remote_code=True, attn_implementation="eager")

In [5]:
jinaai_model = jinaai_model.cuda()

In [7]:
BATCH_SIZE = 32

In [17]:
filters = tables.Filters(3, "blosc:lz4")

In [20]:
with tables.open_file("data_train_embeddings.hdf5", "w") as file:
    array = file.create_carray(file.root, "embeddings", tables.Float32Atom(), shape=(len(train), 1024), filters=filters)
    answers = file.create_carray(file.root, "label", tables.Int8Atom(), shape=(len(train),), filters=filters)

    for i in tqdm.trange(0, len(train), BATCH_SIZE):
        array[i:i + BATCH_SIZE] = jinaai_model.encode(train["text"][i:i + BATCH_SIZE])
        answers[i:i + BATCH_SIZE] = train["label"][i:i + BATCH_SIZE]
        # array.append(jinaai_model.encode(train["text"][i:i + BATCH_SIZE]))
        # answers.append(train["label"][i:i + BATCH_SIZE])
        

  0%|          | 0/3750 [00:00<?, ?it/s]

In [23]:
with tables.open_file("data_test_embeddings.hdf5", "w") as file:
    array = file.create_carray(file.root, "embeddings", tables.Float32Atom(), shape=(len(test), 1024), filters=filters)
    answers = file.create_carray(file.root, "label", tables.Int8Atom(), shape=(len(test),), filters=filters)

    for i in tqdm.trange(0, len(test), BATCH_SIZE):
        array[i:i + BATCH_SIZE] = jinaai_model.encode(test["text"][i:i + BATCH_SIZE])
        answers[i:i + BATCH_SIZE] = test["label"][i:i + BATCH_SIZE]
        # array.append(jinaai_model.encode(train["text"][i:i + BATCH_SIZE]))
        # answers.append(train["label"][i:i + BATCH_SIZE])
        

  0%|          | 0/238 [00:00<?, ?it/s]

In [31]:
train_file, test_file = tables.open_file("data_train_embeddings.hdf5", "r"), tables.open_file("data_test_embeddings.hdf5", "r")

In [19]:
class Datasett(keras.utils.PyDataset):
    def __init__(self, dataset: tables.File, batch_size: int = BATCH_SIZE, shuffle: bool = False, **kwargs):
        super().__init__(**kwargs)
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.rng = np.random.default_rng()

        self.idx = np.arange(len(self.dataset.root["label"]))

        self.on_epoch_end()

    def __len__(self) -> int:
        return (self.idx.shape[0] + self.batch_size - 1) // self.batch_size
    
    def __getitem__(self, idx: int) -> tuple[np.ndarray, np.ndarray]:
        beg_ind = idx * self.batch_size
        end_ind = (idx + 1) * self.batch_size
    
        idx = self.idx[beg_ind:end_ind]

        return self.dataset.root["embeddings"][idx, :], self.dataset.root["label"][idx]

    def on_epoch_end(self) -> None:
        if self.shuffle:
            self.idx = self.rng.choice(self.idx, size=len(self.idx), replace=False)
        
        

In [59]:
train_datasett = Datasett(
    train_file, 
    shuffle=True,
)

test_datasett = Datasett(
    test_file, 
)

In [65]:
inputs = keras.layers.Input((1024,), name="input")
x = keras.layers.Dense(4, name="output")(inputs)

model = keras.Model(inputs=inputs, outputs=x, name="model")

In [66]:
model.compile(
    optimizer="adam",
    metrics=["accuracy"],
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True)
)

In [67]:
model.fit(
    train_datasett,
    validation_data=test_datasett,
    epochs=10
)

Epoch 1/10
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 22s 6ms/step - accuracy: 0.8865 - loss: 0.4718 - val_accuracy: 0.9025 - val_loss: 0.3174
Epoch 2/10
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 22s 6ms/step - accuracy: 0.9065 - loss: 0.2902 - val_accuracy: 0.9068 - val_loss: 0.2877
Epoch 3/10
2655/3750 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.9097 - loss: 0.2739

KeyboardInterrupt: 

In [ ]:
# 0 - политика
# 1 - спорт
# 2 - бизнес
# 3 - технологии

In [39]:
cls_idx_x_names = ["politics", "sports", "business", "tech"]

In [82]:
def inference(textt: str, embedder: torch.nn.Module, classificator: keras.Model) -> str:
    emb = embedder.encode([textt], task="classification")
    pred = classificator.predict(emb, verbose=False)
    pred = np.exp(pred[0])
    pred /= pred.sum()
    return pred, cls_idx_x_names[pred.argmax()]
    

In [74]:
test[52]

{'text': "Restive Maldives eases curfew after rounding up dissidents (AFP) AFP - A curfew in the capital of the Maldives was eased but parliament sessions were put off indefinitely and emergency rule continued following last week's riots, officials and residents said.",
 'label': 0}

In [90]:
inference(
    "Мақомот ва сокинон гуфтанд, ки пас аз ҷамъ кардани мухолифон дар Малдив соати комендантӣ сабуктар шуд (AFP) AFP - Соатҳои комендантӣ дар пойтахти Молдив сабук карда шуд, аммо иҷлосияҳои порлумон ба муддати номуайян мавқуф гузошта шуданд ва тартиботи изтирорӣ дар пайи ошӯбҳои ҳафтаи гузашта идома ёфт.",
    jinaai_model, 
    model,
)

(array([0.70368826, 0.1211346 , 0.09768178, 0.07749539], dtype=float32),
 'politics')

# Qwen

In [4]:
BATCH_SIZE = 4

In [4]:
qwen_tokenizer = transformers.AutoTokenizer.from_pretrained('Qwen/Qwen3-Embedding-8B', padding_side='left')
qwen_model = transformers.AutoModel.from_pretrained('Qwen/Qwen3-Embedding-8B', attn_implementation="sdpa", torch_dtype=torch.float16)

2025-10-16 13:42:55.421289: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-16 13:42:55.423332: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-16 13:42:55.428389: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760586175.439811 3846225 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760586175.442169 3846225 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been regist

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [5]:
qwen_model

Qwen3Model(
  (embed_tokens): Embedding(151665, 4096)
  (layers): ModuleList(
    (0-35): 36 x Qwen3DecoderLayer(
      (self_attn): Qwen3Attention(
        (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
        (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
        (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
        (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
        (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
      )
      (mlp): Qwen3MLP(
        (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
        (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
        (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
      (post_attention_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
    )
  )
  (norm): Qwen3RMSNorm((

In [12]:
qwen_model = qwen_model.cuda()

In [13]:
def add_instruction(text: str):
    return f"Classify text into one of the 4 classes: politics, sports, business, tech. Text: {text}"

In [14]:
filters = tables.Filters(3, "blosc:lz4")

In [17]:
with tables.open_file("data_test_embeddings_qwen.hdf5", "w") as file, torch.no_grad():
    array = file.create_carray(file.root, "embeddings", tables.Float32Atom(), shape=(len(test), 4096), filters=filters)
    answers = file.create_carray(file.root, "label", tables.Int8Atom(), shape=(len(test),), filters=filters)

    for i in tqdm.trange(0, len(test), BATCH_SIZE):
        # array[i:i + BATCH_SIZE] = jinaai_model.encode(test["text"][i:i + BATCH_SIZE])
        # answers[i:i + BATCH_SIZE] = test["label"][i:i + BATCH_SIZE]
        # array.append(jinaai_model.encode(train["text"][i:i + BATCH_SIZE]))
        # answers.append(train["label"][i:i + BATCH_SIZE])
        tokens_and_masks = qwen_tokenizer(
            [add_instruction(one_text) for one_text in test["text"][i:i + BATCH_SIZE]],
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt",
        )
        tokens_and_masks.to(qwen_model.device)
        outputs = qwen_model(**tokens_and_masks)

        array[i:i + BATCH_SIZE] = outputs.last_hidden_state[:, -1].cpu().numpy()
        answers[i:i + BATCH_SIZE] = test["label"][i:i + BATCH_SIZE]
        del tokens_and_masks
        del outputs
        

  0%|          | 0/1900 [00:00<?, ?it/s]

In [18]:
with tables.open_file("data_train_embeddings_qwen.hdf5", "w") as file, torch.no_grad():
    array = file.create_carray(file.root, "embeddings", tables.Float32Atom(), shape=(len(train), 4096), filters=filters)
    answers = file.create_carray(file.root, "label", tables.Int8Atom(), shape=(len(train),), filters=filters)

    for i in tqdm.trange(0, len(train), BATCH_SIZE):
        tokens_and_masks = qwen_tokenizer(
            [add_instruction(one_text) for one_text in train["text"][i:i + BATCH_SIZE]],
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt",
        )
        tokens_and_masks.to(qwen_model.device)
        outputs = qwen_model(**tokens_and_masks)

        array[i:i + BATCH_SIZE] = outputs.last_hidden_state[:, -1].cpu().numpy()
        answers[i:i + BATCH_SIZE] = train["label"][i:i + BATCH_SIZE]
        del tokens_and_masks
        del outputs

  0%|          | 0/30000 [00:00<?, ?it/s]

In [20]:
train_file, test_file = tables.open_file("data_train_embeddings_qwen.hdf5", "r"), tables.open_file("data_test_embeddings_qwen.hdf5", "r")

In [28]:
train_datasett = Datasett(
    train_file, 
    shuffle=True,
    batch_size=64,
)

test_datasett = Datasett(
    test_file, 
    batch_size=64,
)

In [29]:
inputs = keras.layers.Input((4096,), name="input")
x = keras.layers.Dense(4, name="output")(inputs)

model_qwen = keras.Model(inputs=inputs, outputs=x, name="qwen")

In [31]:
model_qwen.compile(
    optimizer="adam",
    metrics=["accuracy"],
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True)
)

In [32]:
model_qwen.fit(
    train_datasett,
    validation_data=test_datasett,
    epochs=10
)

Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 40s 21ms/step - accuracy: 0.9046 - loss: 0.3407 - val_accuracy: 0.9059 - val_loss: 0.3452
Epoch 2/10
 170/1875 ━━━━━━━━━━━━━━━━━━━━ 37s 22ms/step - accuracy: 0.8998 - loss: 0.4136

KeyboardInterrupt: 

In [36]:
def qwen_inference(textt: str, tokenizer, embedder: torch.nn.Module, classificator: keras.Model) -> str:
    with torch.no_grad():
        tokens_and_masks = tokenizer(
            [add_instruction(textt)],
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt",
        )
        tokens_and_masks.to(qwen_model.device)
        
        outputs = embedder(**tokens_and_masks)
        emb = outputs.last_hidden_state[:, -1].cpu().numpy()
        
        pred = classificator.predict(emb, verbose=False)
        pred = np.exp(pred[0])
        pred /= pred.sum()
        return pred, cls_idx_x_names[pred.argmax()]
    

In [40]:
cls_idx_x_names = ["politics", "sports", "business", "tech"]

In [43]:
test[11]

{'text': 'Apple Launches Graphics Software, Video Bundle  LOS ANGELES (Reuters) - Apple Computer Inc.&lt;AAPL.O&gt; on  Tuesday began shipping a new program designed to let users  create real-time motion graphics and unveiled a discount  video-editing software bundle featuring its flagship Final Cut  Pro software.',
 'label': 3}

In [51]:
qwen_inference(
    """
    Новый баскетбольный комплекс площадью более трех тысяч квадратных метров построен в Дальневосточном федеральном университете совместно с ООО «Холдинговая компания "Интеррос"» и АО «Т-Банк». Открыл площадку региональный турнир по баскетболу 3х3, организованный совместно с Ассоциацией студенческого баскетбола (АСБ).

Комплекс находится на месте старой баскетбольной площадки ДВФУ. Среди обновленной инфраструктуры: арена для классического баскетбола, площадки для игр формата 3х3, универсальные поля и трибуны на 250 мест.
    """
    , qwen_tokenizer, 
    qwen_model, 
    model_qwen,
)

(array([7.3686298e-03, 9.8939490e-01, 3.2358817e-03, 6.5295791e-07],
       dtype=float32),
 'sports')